# Download Dataset

Download small tinystories dataset

In [1]:
from datasets import load_dataset

d:\ml\neural-tokenizer-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("roneneldan/TinyStories")

In [3]:
train = dataset['train'].select(range(50000))
valid = dataset['validation'].select(range(5000))

# BPE Tokenizer

In [4]:
SPECIAL_TOKENS = [
    "<pad>",
    "<bos>",
    "<eos>",
]

In [5]:
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFKC
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer

In [6]:
def train_BPE(
        texts: list[str],
        vocab_size: int = 8192,
        output_dir: str = "artifacts/bpe",
) -> Tokenizer:
    tokenizer = Tokenizer(BPE())
    tokenizer.normalizer = NFKC()
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False,)

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        initial_alphabet=ByteLevel.alphabet(),
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )

    tokenizer.train_from_iterator(texts, trainer=trainer)

    path = Path(output_dir)
    path.mkdir(parents=True, exist_ok=True)

    tokenizer.save(str(path / "tokenizer.json"))

    return tokenizer

In [7]:
bpe = train_BPE(train["text"])

In [8]:
print(f"BPE vocabulary size: {bpe.get_vocab_size()}")

BPE vocabulary size: 8192


# Raw Byte Tokenizer

In [9]:
SPECIAL_TOKENS_DIR = {
    "<pad>": 256,
    "<unk>": 257,
    "<bos>": 258,
    "<eos>": 259,
}

In [10]:
VOCAB_SIZE = 260

In [11]:
def encode(text: str) -> list[int]:
    return list(text.encode("utf-8"))

In [12]:
def decode(ids: list[int]) -> str:
    return bytes(ids).decode("utf-8", errors="replace")

# Examples

In [13]:
examples = [
        "The cat is sitting on the mat.",
        "The quantum computer has 10 qubits.",
        "E = mc^2",
        "12345678901234567890",
        "def hello(x): return x + 1",
        "∂²ψ/∂x² + V(x)ψ = Eψ",
        "नमस्ते दुनिया",
        "你好世界",
    ]

In [14]:
op_folder = "output/bpe"

In [15]:
op_path = Path(op_folder)
op_path.mkdir(parents=True, exist_ok=True)

In [16]:
examples_file = op_path/"examples.txt"

In [20]:
f = open(examples_file, "w",  encoding="utf-8")

for text in examples:
        bpe_encoding = bpe.encode(text)
        byte_ids = encode(text)

        f.write("\n" + "=" * 80)
        f.write(f"\nTEXT: {text}")

        f.write("\nBPE:\n")
        f.write(", ".join(bpe_encoding.tokens))
        f.write("\n")
        f.write(", ".join(str(item) for item in bpe_encoding.ids))
        f.write("\n")
        f.write(f"Count: {len(bpe_encoding.ids)}")

        f.write("\nRAW BYTES:")
        f.write(", ".join(str(item) for item in byte_ids))
        f.write("\n")
        f.write(f"Count: {len(byte_ids)}")

        f.write("\nDecode:")
        f.write(decode(byte_ids))

f.close()

In [18]:
def summarize_tokenization(texts, bpe):
    total_chars = 0
    total_bpe = 0
    total_bytes = 0

    for text in texts:
        total_chars += len(text)

        total_bpe += len(
            bpe.encode(text).ids
        )

        total_bytes += len(
            text.encode("utf-8")
        )

    print("\nDataset statistics")
    print("------------------")

    print(f"Characters: {total_chars:,}")
    print(f"BPE tokens: {total_bpe:,}")
    print(f"UTF-8 bytes: {total_bytes:,}")

    print(
        f"BPE / character: "
        f"{total_bpe / total_chars:.4f}"
    )

    print(
        f"Byte / character: "
        f"{total_bytes / total_chars:.4f}"
    )

    print(
        f"BPE compression vs bytes: "
        f"{total_bytes / total_bpe:.2f}x"
    )

In [19]:
summarize_tokenization(valid["text"], bpe)


Dataset statistics
------------------
Characters: 4,063,945
BPE tokens: 978,823
UTF-8 bytes: 4,070,097
BPE / character: 0.2409
Byte / character: 1.0015
BPE compression vs bytes: 4.16x
